# 02. Praproses dan Feature Engineering
Notebook ini mengubah tabel komposisi bersih dari notebook EDA menjadi **data siap latih** untuk model prediksi agent.

**Ide dasarnya.** Satu komposisi berisi 5 agent. Dari satu komposisi kita buat banyak *situasi*: pilih 0 sampai 4 agent sebagai "sudah dipilih pemain", lalu sisanya adalah agent yang harus ditebak. Tiap agent yang tersisa adalah jawaban yang sama-sama benar untuk situasi itu.

**Syarat:** notebook EDA sudah dijalankan sampai selesai, sehingga `hasil/komposisi_valid_2023_2026.csv` ada. Letakkan notebook ini di folder yang sama dengan `vct_2023` ... `vct_2026`.

**Penjelasan blok 1:** Menyiapkan library, folder, tahun training dan testing, serta tabel peran agent (disalin dari notebook EDA kalian). `K_SMOOTH` mengatur seberapa kuat win rate dihaluskan nanti. Tidak ada output.

In [1]:
import json
import numpy as np
import pandas as pd
from itertools import combinations
from pathlib import Path

BASE = Path(".")                  # folder yang berisi vct_2021 ... vct_2026
HASIL = Path("hasil")             # folder hasil dari notebook EDA
TRAIN_YEARS = [2023, 2024, 2025]
TEST_YEARS = [2026]
K_SMOOTH = 10                     # kekuatan penghalusan win rate

def read(year, sub, name):
    return pd.read_csv(BASE / f"vct_{year}" / sub / name, keep_default_na=False, na_values=[''], low_memory=False)

ROLE = {
    # Duelist
    'jett': 'Duelist', 'phoenix': 'Duelist', 'reyna': 'Duelist', 'raze': 'Duelist',
    'yoru': 'Duelist', 'neon': 'Duelist', 'iso': 'Duelist', 'waylay': 'Duelist',
    # Initiator
    'sova': 'Initiator', 'breach': 'Initiator', 'skye': 'Initiator', 'kayo': 'Initiator',
    'fade': 'Initiator', 'gekko': 'Initiator', 'tejo': 'Initiator',
    # Controller
    'brimstone': 'Controller', 'omen': 'Controller', 'viper': 'Controller',
    'astra': 'Controller', 'harbor': 'Controller', 'clove': 'Controller',
    # Sentinel
    'sage': 'Sentinel', 'cypher': 'Sentinel', 'killjoy': 'Sentinel',
    'chamber': 'Sentinel', 'deadlock': 'Sentinel', 'vyse': 'Sentinel',
    # agent baru: isi sendiri setelah dicek
    'miks': 'Controller', 'veto': 'Sentinel',
}

PERAN = ['Duelist', 'Initiator', 'Controller', 'Sentinel']

**Catatan kita:** ...

**Penjelasan blok 2:** Membaca tabel komposisi bersih hasil notebook EDA. Kolom `agents` tersimpan sebagai teks yang dipisah `|`, jadi dipecah kembali menjadi list. Cek bahwa jumlahnya 8.192 dan semuanya berisi 5 agent.

In [2]:
comp = pd.read_csv(HASIL / "komposisi_valid_2023_2026.csv", keep_default_na=False)
comp['agents'] = comp['agents'].str.split('|')      # teks "a|b|c|d|e" kembali menjadi list

print("jumlah komposisi:", len(comp))
print(comp['Tahun'].value_counts().sort_index().to_dict())
assert (comp['agents'].str.len() == 5).all()        # pastikan semua komposisi berisi 5 agent
comp.head(3)

jumlah komposisi: 8192
{2023: 1660, 2024: 2208, 2025: 2552, 2026: 1772}


,Tournament,Stage,Match Type,Match Name,Map,Team,Tahun,jumlah_pemain,agents
0,Champions Tour 2023: Americas Last Chance Qual...,Main Event,Grand Final,KRÜ Esports vs LEVIATÁN,Ascent,KRÜ Esports,2023,5,"[jett, sova, omen, kayo, killjoy]"
1,Champions Tour 2023: Americas Last Chance Qual...,Main Event,Grand Final,KRÜ Esports vs LEVIATÁN,Ascent,LEVIATÁN,2023,5,"[kayo, sova, killjoy, omen, jett]"
2,Champions Tour 2023: Americas Last Chance Qual...,Main Event,Grand Final,KRÜ Esports vs LEVIATÁN,Lotus,KRÜ Esports,2023,5,"[raze, viper, skye, omen, killjoy]"


**Penjelasan blok 3:** Membagi data **menurut waktu**: 2023-2025 untuk training dan 2026 untuk testing, tanpa diacak. Pembagian dilakukan di awal supaya semua langkah sesudahnya (termasuk win rate) hanya memakai data training dan tidak bocor.

In [3]:
train_comp = comp[comp['Tahun'].isin(TRAIN_YEARS)].reset_index(drop=True)
test_comp = comp[comp['Tahun'].isin(TEST_YEARS)].reset_index(drop=True)

print("training:", len(train_comp), "komposisi | testing:", len(test_comp), "komposisi")
assert max(TRAIN_YEARS) < min(TEST_YEARS)           # data uji harus lebih baru dari data latih

training: 6420 komposisi | testing: 1772 komposisi


**Catatan kita:** ...

**Penjelasan blok 4:** Memberi nomor untuk tiap agent, map, dan peran agar bisa dipakai sebagai kolom angka. Di akhir dicek agent yang **tidak pernah muncul di data training** (misalnya agent baru di 2026). Model tidak akan bisa memprediksi agent itu dari data training saja.

In [4]:
AGENTS = sorted({a for l in comp['agents'] for a in l})
MAPS = sorted(comp['Map'].unique())
a_idx = {a: i for i, a in enumerate(AGENTS)}        # nama agent -> nomor
m_idx = {m: i for i, m in enumerate(MAPS)}          # nama map -> nomor
role_of = np.array([PERAN.index(ROLE[a]) for a in AGENTS])   # nomor peran tiap agent

train_agents = {a for l in train_comp['agents'] for a in l}
print("jumlah agent:", len(AGENTS), "| jumlah map:", len(MAPS))
print("agent yang tidak ada di training:", sorted(set(AGENTS) - train_agents))

jumlah agent: 29 | jumlah map: 12
agent yang tidak ada di training: ['miks', 'veto']


**Catatan kita:** ...

**Penjelasan blok 5:** Blok terpenting di notebook ini. Dari tiap komposisi dibuat semua situasi dengan 0 sampai 4 agent yang sudah dipilih (31 situasi per komposisi). Fitur tiap situasi: map, agent yang sudah dipilih, jumlah agent diketahui, dan jumlah tiap peran. `SISA` menyimpan agent-agent yang tersisa, yaitu semua jawaban yang dianggap benar.

In [5]:
NM, NA, NP = len(MAPS), len(AGENTS), len(PERAN)
FITUR = ([f"map_{m}" for m in MAPS] + [f"ada_{a}" for a in AGENTS]
         + ['jumlah_diketahui'] + [f"peran_{p}" for p in PERAN])

def buat_situasi(df):
    X, SISA = [], []
    for map_, ags in zip(df['Map'], df['agents']):
        idx = [a_idx[a] for a in ags]
        for k in range(5):                              # k = jumlah agent yang sudah dipilih (0 sampai 4)
            for known in combinations(idx, k):          # semua cara memilih k agent dari 5
                x = np.zeros(len(FITUR), dtype=np.uint8)
                x[m_idx[map_]] = 1                      # map (one-hot)
                x[NM + NA] = k                          # jumlah agent diketahui
                for j in known:
                    x[NM + j] = 1                       # agent yang sudah dipilih
                    x[NM + NA + 1 + role_of[j]] += 1    # hitung peran dari agent itu
                sisa = np.zeros(NA, dtype=np.uint8)
                sisa[[j for j in idx if j not in known]] = 1   # agent sisa = jawaban yang benar
                X.append(x)
                SISA.append(sisa)
    return np.array(X), np.array(SISA)

X_train_sit, sisa_train = buat_situasi(train_comp)
X_test_sit, sisa_test = buat_situasi(test_comp)

print("fitur:", len(FITUR))
print("situasi training:", X_train_sit.shape, "| situasi testing:", X_test_sit.shape)
print("situasi per jumlah agent diketahui (training):",
      pd.Series(X_train_sit[:, NM + NA]).value_counts().sort_index().to_dict())

fitur: 46
situasi training: (199020, 46) | situasi testing: (54932, 46)
situasi per jumlah agent diketahui (training): {0: 6420, 1: 32100, 2: 64200, 3: 64200, 4: 32100}


**Catatan kita:** ...

**Penjelasan blok 6:** Mengubah satu baris angka kembali menjadi bahasa manusia, supaya kita yakin pembentukan situasinya benar. Contoh ini harus masuk akal: agent sisa tidak boleh sama dengan agent yang sudah dipilih, dan hitungan peran harus cocok dengan agent yang sudah dipilih.

In [6]:
def jelaskan(X, SISA, i):
    map_ = MAPS[int(np.argmax(X[i, :NM]))]
    diketahui = [AGENTS[j] for j in np.nonzero(X[i, NM:NM + NA])[0]]
    sisa = [AGENTS[j] for j in np.nonzero(SISA[i])[0]]
    return map_, diketahui, sisa

i = np.where(X_test_sit[:, NM + NA] == 2)[0][0]      # situasi pertama dengan 2 agent diketahui
m, diketahui, sisa = jelaskan(X_test_sit, sisa_test, i)
print("Map                  :", m)
print("Agent sudah dipilih  :", diketahui)
print("Jawaban benar (sisa) :", sisa)
print("Hitungan peran       :", dict(zip(PERAN, X_test_sit[i, NM + NA + 1:].tolist())))

Map                  : Abyss
Agent sudah dipilih  : ['omen', 'sova']
Jawaban benar (sisa) : ['astra', 'jett', 'kayo']
Hitungan peran       : {'Duelist': 0, 'Initiator': 1, 'Controller': 1, 'Sentinel': 0}


**Catatan kita:** ...

**Penjelasan blok 7:** Model klasifikasi biasa hanya menerima satu label per baris, padahal satu situasi punya beberapa jawaban benar. Jadi satu situasi diperbanyak menjadi satu baris untuk tiap agent sisa. Data testing sengaja tidak diperbanyak karena nanti dinilai memakai seluruh agent sisa sebagai jawaban benar.

In [7]:
baris, label = np.nonzero(sisa_train)        # tiap pasangan (situasi, agent sisa) = 1 sampel latih
X_train = X_train_sit[baris]
y_train = label

print("sampel latih:", X_train.shape, "| label:", y_train.shape)
print("rata-rata jawaban benar per situasi:", round(len(baris) / len(X_train_sit), 2))
print("5 label terbanyak:", pd.Series([AGENTS[j] for j in y_train]).value_counts().head(5).to_dict())

sampel latih: (513600, 46) | label: (513600,)
rata-rata jawaban benar per situasi: 2.58
5 label terbanyak: {'omen': 61376, 'viper': 50512, 'raze': 39248, 'sova': 39104, 'jett': 37840}


**Catatan kita:** ...

**Penjelasan blok 8:** Menghitung win rate tiap agent di tiap map **hanya dari data training** (2023-2025) agar tidak bocor. Win rate dihaluskan dengan menambahkan `K_SMOOTH` pertandingan bayangan bernilai 50%: kombinasi dengan sampel kecil ditarik mendekati 50%, sedangkan yang sampelnya besar hampir tidak berubah. Bandingkan kolom `win_rate_mentah` dan `win_rate_halus`.

In [8]:
tpa = pd.concat([read(y, "agents", "teams_picked_agents.csv") for y in TRAIN_YEARS], ignore_index=True)
tpa = tpa[(tpa['Stage'] != 'All Stages') & (tpa['Match Type'] != 'All Match Types') & (tpa['Map'] != 'All Maps')]

wr = tpa.groupby(['Map', 'Agent'])[['Total Wins By Map', 'Total Loss By Map']].sum()
wr['dimainkan'] = wr.sum(axis=1)
wr['win_rate_mentah'] = wr['Total Wins By Map'] / wr['dimainkan']
wr['win_rate_halus'] = (wr['Total Wins By Map'] + K_SMOOTH * 0.5) / (wr['dimainkan'] + K_SMOOTH)
wr = wr.reset_index()

print("kombinasi agent-map:", len(wr))
wr.sort_values('win_rate_halus', ascending=False).head(10)

kombinasi agent-map: 292


,Map,Agent,Total Wins By Map,Total Loss By Map,dimainkan,win_rate_mentah,win_rate_halus
285,Sunset,sage,31,14,45,0.688889,0.654545
100,Corrode,kayo,15,6,21,0.714286,0.645161
230,Pearl,phoenix,21,11,32,0.656250,0.619048
1,Abyss,breach,14,7,21,0.666667,0.612903
185,Icebox,vyse,6,2,8,0.750000,0.611111
27,Ascent,chamber,20,11,31,0.645161,0.609756
77,Breeze,astra,20,12,32,0.625000,0.595238
68,Bind,reyna,11,6,17,0.647059,0.592593
111,Corrode,waylay,18,11,29,0.620690,0.589744
125,Fracture,neon,71,49,120,0.591667,0.584615


**Catatan kita:** ...

**Penjelasan blok 9:** Menyimpan data siap latih (`data_model.npz`), daftar agent/map/fitur (`meta_model.json`), tabel win rate, dan tabel peran. Notebook model nanti tinggal memuat file-file ini, dan client program juga memakai tabel win rate dan tabel peran. File `.npz` cukup besar, jadi sebaiknya masuk `.gitignore`.

In [9]:
HASIL.mkdir(exist_ok=True)

np.savez_compressed(HASIL / "data_model.npz",
                    X_train=X_train, y_train=y_train,
                    X_test=X_test_sit, sisa_test=sisa_test)

meta = {"AGENTS": AGENTS, "MAPS": MAPS, "PERAN": PERAN, "FITUR": FITUR,
        "ROLE": {a: ROLE[a] for a in AGENTS}}
with open(HASIL / "meta_model.json", "w") as f:
    json.dump(meta, f, indent=2)

wr.to_csv(HASIL / "winrate_halus_train.csv", index=False)
pd.DataFrame({"agent": AGENTS, "peran": [ROLE[a] for a in AGENTS]}).to_csv(HASIL / "peran_agent.csv", index=False)

print("tersimpan:", sorted(p.name for p in HASIL.iterdir()))

tersimpan: ['data_model.npz', 'komposisi_valid_2023_2026.csv', 'meta_model.json', 'peran_agent.csv', 'winrate_agent_per_map.csv', 'winrate_halus_train.csv']


## Kesimpulan praproses
Tulis dengan kata kalian sendiri:
1. Berapa situasi dan sampel latih yang terbentuk, dan bagaimana sebarannya per jumlah agent diketahui?
2. Fitur apa yang kita pakai dan kenapa?
3. Apa dampak agent yang tidak ada di data training?